# Khám phá bộ dữ liệu Legal Machine Translation

Notebook này giúp đọc và hiểu nhanh bộ dữ liệu dịch máy pháp lý Anh–Việt. Các mục tiêu chính:

- đọc đúng CSV có trường văn bản nhiều dòng;
- kiểm tra schema, dữ liệu thiếu và trùng lặp;
- khảo sát phân bố lĩnh vực và độ dài tài liệu;
- xem các cặp tài liệu Anh–Việt mẫu;
- nhận diện các vấn đề cần xử lý trước khi huấn luyện.

> Notebook chỉ đọc dữ liệu, không sửa các tệp CSV gốc.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print(f"pandas={pd.__version__} | numpy={np.__version__}")

pandas=2.3.3 | numpy=2.3.3


## 1. Xác định đường dẫn dữ liệu

Cell dưới tự tìm project root nên notebook có thể được chạy từ thư mục gốc hoặc từ thư mục `notebooks`.

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "instruction.md").exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy data/instruction.md. Hãy mở notebook bên trong project này."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INSTRUCTION_PATH = PROJECT_ROOT / "data" / "instruction.md"
TRAIN_PATH = PROJECT_ROOT / "data" / "train" / "en-vi-train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "test" / "en-vi-public_test.csv"

paths = {
    "project_root": PROJECT_ROOT,
    "instruction": INSTRUCTION_PATH,
    "train": TRAIN_PATH,
    "public_test": TEST_PATH,
}
display(pd.Series(paths, name="path").to_frame())
assert all(path.exists() for path in paths.values()), "Có đường dẫn dữ liệu không tồn tại."

,path
project_root,D:\MyProject\NLP
instruction,D:\MyProject\NLP\data\instruction.md
train,D:\MyProject\NLP\data\train\en-vi-train.csv
public_test,D:\MyProject\NLP\data\test\en-vi-public_test.csv


## 2. Đọc mô tả cuộc thi

Hiển thị phần đầu của instruction để nhắc lại mục tiêu và ràng buộc trước khi phân tích dữ liệu.

In [3]:
instruction = INSTRUCTION_PATH.read_text(encoding="utf-8")
instruction_preview = "\n".join(instruction.splitlines()[:55])
display(Markdown(instruction_preview))

# Legal Machine Translation Shared Task

## 1. Important Dates

| Date | Milestone |
|---|---|
| **September 18** | Training Data release |
| **September 25** | Public Test |
| **October 9** | Private Test |
| **October 15** | Result announcement |
| **October 25** | Paper submission |
| **November 5** | Acceptance notification |
| **November 12** | Camera-ready / Proceedings |
| **November 15** | Workshop |

---

## 2. Task Description

Machine Translation (MT) trong **lĩnh vực pháp lý (Legal Domain)** là một bài toán khó vì văn bản pháp luật yêu cầu:

- Độ chính xác cao.
- Sử dụng thuật ngữ pháp lý nhất quán.
- Bảo toàn chính xác ý nghĩa pháp lý của văn bản gốc.

Các văn bản pháp luật thường có:

- Câu dài và cấu trúc phức tạp.
- Nhiều thuật ngữ chuyên ngành.
- Các biểu đạt có ý nghĩa phụ thuộc mạnh vào ngữ cảnh pháp lý.
- Các điều khoản, tham chiếu chéo và cách diễn đạt mang tính hình thức.
- Một thuật ngữ có thể cần được dịch nhất quán xuyên suốt toàn bộ văn bản.

Việc dịch không nhất quán hoặc dịch sai thuật ngữ có thể gây **mơ hồ hoặc làm thay đổi ý nghĩa pháp lý**.

Bài toán trở nên khó hơn khi sử dụng các mô hình tương đối nhỏ trong điều kiện **tài nguyên tính toán hạn chế**.

---

## 3. Main Challenges

### 3.1. Legal Terminology

Văn bản pháp luật chứa nhiều thuật ngữ và biểu đạt chuyên ngành.

Một thuật ngữ pháp lý có thể có ý nghĩa khác với cách sử dụng trong ngôn ngữ thông thường.

**Yêu cầu:** Hệ thống phải dịch đúng thuật ngữ theo **ngữ nghĩa pháp lý**.

### 3.2. Terminology Consistency

Cùng một thuật ngữ hoặc khái niệm pháp lý cần được dịch **nhất quán** trong toàn bộ văn bản.

Ví dụ, nếu một thuật ngữ được dịch theo một cách ở phần đầu thì không nên tự ý thay đổi cách dịch ở các phần sau.


## 3. Đọc train và public test

Các ô `english_markdown` và `vietnamese_markdown` chứa xuống dòng và được quote trong CSV. Vì vậy cần dùng CSV parser như `pandas.read_csv`, không đọc từng dòng bằng `readlines()` hoặc `head`.

In [4]:
EXPECTED_COLUMNS = [
    "sample_id",
    "category",
    "english_path",
    "vietnamese_path",
    "english_markdown",
    "vietnamese_markdown",
]


def load_parallel_csv(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(
        path,
        encoding="utf-8",
        keep_default_na=False,
        low_memory=False,
    )
    missing_columns = set(EXPECTED_COLUMNS) - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Thiếu các cột: {sorted(missing_columns)}")
    return frame[EXPECTED_COLUMNS].copy()


train = load_parallel_csv(TRAIN_PATH)
public_test = load_parallel_csv(TEST_PATH)

overview = pd.DataFrame(
    {
        "split": ["train", "public_test"],
        "documents": [len(train), len(public_test)],
        "columns": [train.shape[1], public_test.shape[1]],
        "memory_MB": [
            train.memory_usage(deep=True).sum() / 1024**2,
            public_test.memory_usage(deep=True).sum() / 1024**2,
        ],
    }
)
display(overview)
display(train.head(3)[["sample_id", "category", "english_path", "vietnamese_path"]])

,split,documents,columns,memory_MB
0,train,854,6,157.55
1,public_test,50,6,8.97


,sample_id,category,english_path,vietnamese_path
0,Bat_dong_san/1,Bat_dong_san,en/Bat_dong_san/1.md,vi/Bat_dong_san/1.md
1,Bat_dong_san/10,Bat_dong_san,en/Bat_dong_san/10.md,vi/Bat_dong_san/10.md
2,Bat_dong_san/104,Bat_dong_san,en/Bat_dong_san/104.md,vi/Bat_dong_san/104.md


## 4. Tạo đặc trưng mô tả tài liệu

Các thống kê token ở đây chỉ là **xấp xỉ theo từ tách bằng khoảng trắng**, chưa phải token của pretrained model. Khi chọn model cần đo lại bằng tokenizer tương ứng.

In [5]:
def add_document_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for language, column in [
        ("en", "english_markdown"),
        ("vi", "vietnamese_markdown"),
    ]:
        text = result[column].astype(str)
        result[f"{language}_chars"] = text.str.len()
        result[f"{language}_lines"] = text.str.count(r"\n") + 1
        result[f"{language}_words_approx"] = text.str.count(r"\S+")
        result[f"{language}_paragraphs"] = text.str.count(r"(?:\r?\n){2,}") + 1

    result["char_ratio_en_vi"] = (
        result["en_chars"] / result["vi_chars"].replace(0, np.nan)
    )
    result["line_ratio_en_vi"] = (
        result["en_lines"] / result["vi_lines"].replace(0, np.nan)
    )
    return result


train_eda = add_document_features(train)
test_eda = add_document_features(public_test)
display(train_eda.head(3)[[
    "sample_id", "category", "en_chars", "vi_chars",
    "en_lines", "vi_lines", "char_ratio_en_vi"
]])

,sample_id,category,en_chars,vi_chars,en_lines,vi_lines,char_ratio_en_vi
0,Bat_dong_san/1,Bat_dong_san,24515,20770,417,391,1.18
1,Bat_dong_san/10,Bat_dong_san,13297,12443,215,167,1.07
2,Bat_dong_san/104,Bat_dong_san,35600,32793,556,540,1.09


## 5. Kiểm tra chất lượng cơ bản

Đặc biệt chú ý duplicate: nếu chia train/dev sau khi chia nhỏ document, các đoạn của cùng tài liệu có thể xuất hiện ở cả hai phía và làm metric cao giả tạo.

In [6]:
def quality_report(frame: pd.DataFrame, split_name: str) -> dict:
    empty_mask = frame[EXPECTED_COLUMNS].apply(
        lambda column: column.astype(str).str.strip().eq("")
    )
    return {
        "split": split_name,
        "rows": len(frame),
        "duplicate_sample_id": int(frame["sample_id"].duplicated(keep=False).sum()),
        "duplicate_EN_documents": int(frame["english_markdown"].duplicated(keep=False).sum()),
        "duplicate_VI_documents": int(frame["vietnamese_markdown"].duplicated(keep=False).sum()),
        "duplicate_parallel_pairs": int(
            frame.duplicated(["english_markdown", "vietnamese_markdown"], keep=False).sum()
        ),
        "rows_with_any_empty_field": int(empty_mask.any(axis=1).sum()),
    }


quality = pd.DataFrame([
    quality_report(train, "train"),
    quality_report(public_test, "public_test"),
])
display(quality)

train_en = set(train["english_markdown"])
train_vi = set(train["vietnamese_markdown"])
cross_split = pd.Series({
    "exact_EN_overlap": int(public_test["english_markdown"].isin(train_en).sum()),
    "exact_VI_overlap": int(public_test["vietnamese_markdown"].isin(train_vi).sum()),
})
display(cross_split.to_frame("count"))

,split,rows,duplicate_sample_id,duplicate_EN_documents,duplicate_VI_documents,duplicate_parallel_pairs,rows_with_any_empty_field
0,train,854,0,34,36,34,0
1,public_test,50,0,0,0,0,0


,count
exact_EN_overlap,0
exact_VI_overlap,0


### Xem các nhóm trùng trong train

In [7]:
duplicate_pairs = (
    train.loc[
        train.duplicated(["english_markdown", "vietnamese_markdown"], keep=False),
        ["sample_id", "category", "english_markdown", "vietnamese_markdown"],
    ]
    .assign(
        en_preview=lambda data: data["english_markdown"].str.replace(r"\s+", " ", regex=True).str[:100],
        vi_preview=lambda data: data["vietnamese_markdown"].str.replace(r"\s+", " ", regex=True).str[:100],
    )
    .drop(columns=["english_markdown", "vietnamese_markdown"])
    .sort_values(["en_preview", "sample_id"])
)
display(duplicate_pairs.head(20))

,sample_id,category,en_preview,vi_preview
380,So_huu_tri_tue/15,So_huu_tri_tue,GOVERNMENT OF VIETNAM SOCIALIST REPUBLIC OF VIETNAM ------- Independence - Freedom – Happiness -----,CHÍNH PHỦ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM ------- Độc lập - Tự do - Hạnh phúc --------------- Số:
803,Vi_pham_hanh_chinh/46,Vi_pham_hanh_chinh,GOVERNMENT OF VIETNAM SOCIALIST REPUBLIC OF VIETNAM ------- Independence - Freedom – Happiness -----,CHÍNH PHỦ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM ------- Độc lập - Tự do - Hạnh phúc --------------- Số:
403,So_huu_tri_tue/3,So_huu_tri_tue,GOVERNMENT OF VIETNAM THE SOCIALIST REPUBLIC OF VIETNAM ------- Independence – Freedom – Happiness -,CHÍNH PHỦ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM ------- Độc lập - Tự do - Hạnh phúc --------------- Số:
672,Vi_pham_hanh_chinh/15,Vi_pham_hanh_chinh,GOVERNMENT OF VIETNAM THE SOCIALIST REPUBLIC OF VIETNAM ------- Independence – Freedom – Happiness -,CHÍNH PHỦ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM ------- Độc lập - Tự do - Hạnh phúc --------------- Số:
185,Quyen_dan_su/153,Quyen_dan_su,NATIONAL ASSEMBLY SOCIALIST REPUBLIC OF VIETNAM -------- Independence - Freedom - Happiness --------,QUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM -------- Độc lập - Tự do - Hạnh phúc --------------- Luậ
452,Thu_tuc_to_tung/138,Thu_tuc_to_tung,NATIONAL ASSEMBLY SOCIALIST REPUBLIC OF VIETNAM -------- Independence - Freedom - Happiness --------,QUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM -------- Độc lập - Tự do - Hạnh phúc --------------- Luậ
600,Trach_nhiem_hinh_su/47,Trach_nhiem_hinh_su,THE COUNCIL OF JUDGES SOCIALIST REPUBLIC OF VIETNAM THE SUPREME PEOPLE’S Independence - Freedom – Ha,HỘI ĐỒNG THẨM PHÁN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM TÒA ÁN NHÂN DÂN TỐI Độc lập - Tự do - Hạnh phú
601,Trach_nhiem_hinh_su/48,Trach_nhiem_hinh_su,THE COUNCIL OF JUDGES SOCIALIST REPUBLIC OF VIETNAM THE SUPREME PEOPLE’S Independence - Freedom – Ha,HỘI ĐỒNG THẨM PHÁN CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM TÒA ÁN NHÂN DÂN TỐI Độc lập - Tự do - Hạnh phú
590,Trach_nhiem_hinh_su/35,Trach_nhiem_hinh_su,THE GOVERNMENT OF THE SOCIALIST REPUBLIC OF VIETNAM VIETNAM Independence – Freedom – Happiness -----,CHÍNH PHỦ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM ------- Độc lập - Tự do - Hạnh phúc --------------- Số:
841,Vi_pham_hanh_chinh/75,Vi_pham_hanh_chinh,THE GOVERNMENT OF THE SOCIALIST REPUBLIC OF VIETNAM VIETNAM Independence – Freedom – Happiness -----,CHÍNH PHỦ CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM ------- Độc lập - Tự do - Hạnh phúc --------------- Số:


## 6. Phân bố lĩnh vực

In [8]:
category_distribution = (
    pd.concat(
        [
            train["category"].value_counts().rename("train"),
            public_test["category"].value_counts().rename("public_test"),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
category_distribution["train_%"] = 100 * category_distribution["train"] / len(train)
category_distribution["public_test_%"] = (
    100 * category_distribution["public_test"] / len(public_test)
)
category_distribution = category_distribution.sort_values("train", ascending=False)
display(category_distribution.round({"train_%": 1, "public_test_%": 1}))

,train,public_test,train_%,public_test_%
category,,,,
Vi_pham_hanh_chinh,213,12,24.90,24.00
Quyen_dan_su,197,12,23.10,24.00
Trach_nhiem_hinh_su,119,7,13.90,14.00
So_huu_tri_tue,92,5,10.80,10.00
Bat_dong_san,78,5,9.10,10.00
Dich_vu_phap_ly,78,5,9.10,10.00
Thu_tuc_to_tung,77,4,9.00,8.00


## 7. Phân bố độ dài tài liệu

Các percentile giúp quyết định chiến lược chunking và giới hạn context.

In [9]:
LENGTH_COLUMNS = [
    "en_chars", "vi_chars",
    "en_words_approx", "vi_words_approx",
    "en_lines", "vi_lines",
]
PERCENTILES = [0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]


def length_summary(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    summary = frame[LENGTH_COLUMNS].quantile(PERCENTILES).T
    summary.columns = [f"p{int(p * 100):02d}" for p in PERCENTILES]
    summary.insert(0, "split", split_name)
    return summary


length_stats = pd.concat(
    [length_summary(train_eda, "train"), length_summary(test_eda, "public_test")]
)
display(length_stats.round(0).astype(int, errors="ignore"))

,split,p00,p25,p50,p75,p90,p95,p99,p100
en_chars,train,1213,14283,29166,60146,110628,163118,355341,596720
vi_chars,train,1049,12407,26208,54038,96047,157183,331994,544262
en_words_approx,train,110,2010,4293,8954,16452,24874,52624,90095
vi_words_approx,train,141,2580,5496,11442,20754,32575,69721,118145
en_lines,train,28,229,490,1026,1814,2700,6133,10643
vi_lines,train,32,223,474,973,1715,2623,5812,9675
en_chars,public_test,1431,12683,28438,61973,90057,161090,367867,486366
vi_chars,public_test,1393,11034,24500,54460,82363,141027,311715,405841
en_words_approx,public_test,144,1708,4230,9059,13562,24319,54213,71321
vi_words_approx,public_test,175,2274,5189,11379,17495,29617,70156,91610


In [10]:
largest_documents = (
    train_eda.nlargest(10, "en_chars")[
        ["sample_id", "category", "en_chars", "vi_chars", "en_words_approx", "vi_words_approx"]
    ]
)
display(largest_documents)

,sample_id,category,en_chars,vi_chars,en_words_approx,vi_words_approx
63,Bat_dong_san/7,Bat_dong_san,596720,544262,90095,118145
38,Bat_dong_san/22,Bat_dong_san,580526,541512,89689,117643
511,Thu_tuc_to_tung/63,Thu_tuc_to_tung,580452,512831,89270,111311
53,Bat_dong_san/45,Bat_dong_san,577722,540025,89373,117267
536,Trach_nhiem_hinh_su/12,Trach_nhiem_hinh_su,565070,532954,82548,114172
695,Vi_pham_hanh_chinh/189,Vi_pham_hanh_chinh,399148,334470,61160,67619
832,Vi_pham_hanh_chinh/56,Vi_pham_hanh_chinh,399116,354980,60813,76460
723,Vi_pham_hanh_chinh/28,Vi_pham_hanh_chinh,372068,319817,55969,67328
645,Vi_pham_hanh_chinh/109,Vi_pham_hanh_chinh,359778,328077,55342,69042
54,Bat_dong_san/46,Bat_dong_san,351407,337120,49231,75595


## 8. Kiểm tra tỷ lệ độ dài Anh–Việt

Tỷ lệ quá lệch có thể là tín hiệu căn chỉnh document không tốt, bản dịch thiếu đoạn hoặc có phụ lục chỉ xuất hiện ở một ngôn ngữ. Đây chỉ là tín hiệu để kiểm tra, không tự động kết luận là dữ liệu sai.

In [11]:
ratio_summary = pd.concat(
    {
        "train": train_eda["char_ratio_en_vi"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]),
        "public_test": test_eda["char_ratio_en_vi"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]),
    },
    axis=1,
)
display(ratio_summary)

ratio_low, ratio_high = train_eda["char_ratio_en_vi"].quantile([0.01, 0.99])
ratio_outliers = train_eda.loc[
    ~train_eda["char_ratio_en_vi"].between(ratio_low, ratio_high),
    ["sample_id", "category", "en_chars", "vi_chars", "char_ratio_en_vi"],
].sort_values("char_ratio_en_vi")
display(ratio_outliers)

,train,public_test
count,854.00,50.00
mean,1.12,1.10
std,0.14,0.16
min,0.54,0.50
1%,0.77,0.67
5%,0.96,0.88
50%,1.12,1.12
95%,1.27,1.21
99%,1.40,1.52
max,2.63,1.80


,sample_id,category,en_chars,vi_chars,char_ratio_en_vi
103,Dich_vu_phap_ly/21,Dich_vu_phap_ly,6031,11096,0.54
163,Quyen_dan_su/111,Quyen_dan_su,27329,47224,0.58
350,Quyen_dan_su/93,Quyen_dan_su,10046,17260,0.58
269,Quyen_dan_su/325,Quyen_dan_su,4610,7325,0.63
112,Dich_vu_phap_ly/32,Dich_vu_phap_ly,9370,14757,0.63
271,Quyen_dan_su/329,Quyen_dan_su,42214,58218,0.73
95,Dich_vu_phap_ly/134,Dich_vu_phap_ly,16432,22462,0.73
666,Vi_pham_hanh_chinh/14,Vi_pham_hanh_chinh,28044,37471,0.75
683,Vi_pham_hanh_chinh/172,Vi_pham_hanh_chinh,23278,30270,0.77
724,Vi_pham_hanh_chinh/281,Vi_pham_hanh_chinh,1750,1227,1.43


## 9. Xem mẫu song ngữ

Có thể thay `sample_id` hoặc gọi `show_parallel_sample(train, row_index=...)` để xem tài liệu khác. Chỉ hiển thị preview để notebook không bị nặng.

In [12]:
def compact_preview(text: str, max_chars: int = 1_500) -> str:
    text = str(text).replace("\r\n", "\n").strip()
    return text if len(text) <= max_chars else text[:max_chars] + "\n… [đã rút gọn]"


def show_parallel_sample(
    frame: pd.DataFrame,
    *,
    row_index: int = 0,
    sample_id: str | None = None,
    max_chars: int = 1_500,
) -> None:
    if sample_id is not None:
        matches = frame.loc[frame["sample_id"] == sample_id]
        if matches.empty:
            raise KeyError(f"Không tìm thấy sample_id={sample_id!r}")
        row = matches.iloc[0]
    else:
        row = frame.iloc[row_index]

    display(Markdown(
        f"### `{row['sample_id']}` — `{row['category']}`\n"
        f"- English path: `{row['english_path']}`\n"
        f"- Vietnamese path: `{row['vietnamese_path']}`"
    ))
    display(Markdown("#### English preview\n\n```text\n" + compact_preview(row["english_markdown"], max_chars) + "\n```"))
    display(Markdown("#### Vietnamese preview\n\n```text\n" + compact_preview(row["vietnamese_markdown"], max_chars) + "\n```"))


show_parallel_sample(train, row_index=0)

### `Bat_dong_san/1` — `Bat_dong_san`
- English path: `en/Bat_dong_san/1.md`
- Vietnamese path: `vi/Bat_dong_san/1.md`

#### English preview

```text
GOVERNMENT OF VIETNAM                           THE SOCIALIST REPUBLIC OF VIETNAM
                --------                                   Independence - Freedom – Happiness
                                                                      ---------------
                No. 326/2026/ND-CP                                                          Hanoi, August 19, 2026


                                                       DECREE

                                         ON LOCATION IDENTIFICATION

  Pursuant to the Law on Government Organization No. 63/2025/QH15;

  Pursuant to the Law on Local Government Organization No. 72/2025/QH15;

  Pursuant to the Law on Digital Data No. 60/2024/QH15;

  Pursuant to the Law on Land No. 31/2024/QH15;

  Pursuant to the Law on Topographic and Cartographic No. 27/2018/QH14;

  Pursuant to the Law on Personal Data Protection No. 91/2025/QH15;

  Pursuant to the Law on Promulgation of Legislative Documents No. 64/2025/QH15 amended by the Law No.
  87/2025/QH15;

  At the request of the Minister of Public Security;

  The Government promulgates the Decree on location identification.

  Chapter I

                                             GENERAL REGULATIONS

  Article 1. Scope and regulated entities

  1. This Decree promulgates the identification of locations within the territory of the Socialist Republic of
  Vietnam (excluding those under the List of state secrets); the development, management, connection, sharing,
  and use of
… [đã rút gọn]
```

#### Vietnamese preview

```text
CHÍNH PHỦ                            CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
                      -------                                 Độc lập - Tự do - Hạnh phúc
                                                                      ---------------
                Số: 326/2026/NĐ-CP                                             Hà Nội, ngày 19 tháng 8 năm 2026


                                                   NGHỊ ĐỊNH

                                      QUY ĐỊNH VỀ ĐỊNH DANH ĐỊA ĐIỂM

  Căn cứ Luật Tổ chức Chính phủ số 63/2025/QH15;

  Căn cứ Luật Tổ chức chính quyền địa phương số 72/2025/QH15;

  Căn cứ Luật Dữ liệu số 60/2024/QH15;

  Căn cứ Luật Đất đai số 31/2024/QH15;

  Căn cứ Luật Đo đạc và Bản đồ số 27/2018/QH14;

  Căn cứ Luật Bảo vệ dữ liệu cá nhân số 91/2025/QH15;

  Căn cứ Luật Ban hành văn bản quy phạm pháp luật số 64/2025/QH15 được sửa đổi, bổ sung bởi Luật số
  87/2025/QH15;

  Theo đề nghị của Bộ trưởng Bộ Công an;

  Chính phủ ban hành Nghị định quy định về định danh địa điểm.

  Chương I

                                               QUY ĐỊNH CHUNG

  Điều 1. Phạm vi điều chỉnh, đối tượng áp dụng

  1. Nghị định này quy định về việc định danh đối với các địa điểm trong lãnh thổ nước Cộng hòa xã hội chủ
  nghĩa Việt Nam (trừ các đối tượng thuộc Danh mục bí mật nhà nước); việc xây dựng, quản lý, kết nối, chia sẻ và
  khai thác Cơ sở dữ liệu định danh địa điểm; quyền, nghĩa vụ và trách nhiệm của cơ quan, tổ chức, cá nhân có
  liên quan.

  2. Nghị định này
… [đã rút gọn]
```

## 10. Dấu hiệu cấu trúc pháp lý

Thống kê sơ bộ các dòng bắt đầu bằng `Article`/`Điều`, các khoản đánh số và HTML. Kết quả giúp thiết kế bộ segmenter ở bước tiếp theo.

In [13]:
STRUCTURE_PATTERNS = {
    "en_article_lines": ("english_markdown", r"(?im)^\s*Article\s+\d+"),
    "vi_article_lines": ("vietnamese_markdown", r"(?im)^\s*Điều\s+\d+"),
    "en_numbered_lines": ("english_markdown", r"(?m)^\s*\d+[.)]\s+"),
    "vi_numbered_lines": ("vietnamese_markdown", r"(?m)^\s*\d+[.)]\s+"),
    "en_markdown_headings": ("english_markdown", r"(?m)^#{1,6}\s+"),
    "vi_markdown_headings": ("vietnamese_markdown", r"(?m)^#{1,6}\s+"),
    "en_html_tags": ("english_markdown", r"<[^>]+>"),
    "vi_html_tags": ("vietnamese_markdown", r"<[^>]+>"),
}

structure_stats = {}
for feature, (column, pattern) in STRUCTURE_PATTERNS.items():
    counts = train[column].str.count(pattern)
    structure_stats[feature] = {
        "documents_with_marker": int(counts.gt(0).sum()),
        "total_markers": int(counts.sum()),
        "median_if_present": float(counts[counts.gt(0)].median()) if counts.gt(0).any() else 0,
    }

display(pd.DataFrame(structure_stats).T)

,documents_with_marker,total_markers,median_if_present
en_article_lines,775.00,"25,153.00",19.00
vi_article_lines,779.00,"24,819.00",19.00
en_numbered_lines,819.00,"80,062.00",51.00
vi_numbered_lines,790.00,"77,864.00",51.00
en_markdown_headings,0.00,0.00,0.00
vi_markdown_headings,0.00,0.00,0.00
en_html_tags,8.00,17.00,1.00
vi_html_tags,1.00,7.00,7.00


## 11. Ước lượng nhu cầu chunking

Đây là ước lượng theo số từ, không thay thế tokenizer thật. Mục đích là chứng minh phần lớn document không thể đưa nguyên khối vào một model MT thông thường.

In [14]:
for split_name, frame in [("train", train_eda), ("public_test", test_eda)]:
    chunk_need = pd.Series({
        "documents": len(frame),
        "EN > 512 approximate words": int(frame["en_words_approx"].gt(512).sum()),
        "VI > 512 approximate words": int(frame["vi_words_approx"].gt(512).sum()),
        "EN median approximate chunks@400 words": float(np.ceil(frame["en_words_approx"] / 400).median()),
        "VI median approximate chunks@400 words": float(np.ceil(frame["vi_words_approx"] / 400).median()),
    }, name=split_name)
    display(chunk_need.to_frame())

,train
documents,854.00
EN > 512 approximate words,815.00
VI > 512 approximate words,824.00
EN median approximate chunks@400 words,11.00
VI median approximate chunks@400 words,14.00


,public_test
documents,50.00
EN > 512 approximate words,48.00
VI > 512 approximate words,48.00
EN median approximate chunks@400 words,11.00
VI median approximate chunks@400 words,13.50


## 12. Kết luận tự động từ lần chạy hiện tại

In [15]:
train_duplicate_pairs = train.duplicated(
    ["english_markdown", "vietnamese_markdown"], keep=False
).sum()
median_en_chars = int(train_eda["en_chars"].median())
max_en_chars = int(train_eda["en_chars"].max())
largest_category = train["category"].value_counts().idxmax()

display(Markdown(f"""
### Những điều cần nhớ

- Train có **{len(train):,}** document song ngữ; public test có **{len(public_test):,}** document.
- Độ dài tiếng Anh median là **{median_en_chars:,} ký tự**; document lớn nhất có **{max_en_chars:,} ký tự**.
- Có **{int(train_duplicate_pairs):,} lượt document** thuộc các nhóm cặp song ngữ trùng hoàn toàn. Nên deduplicate trước khi chia train/dev.
- Lĩnh vực lớn nhất trong train là **`{largest_category}`**. Khi tạo dev split nên stratify theo `category`.
- Train và public test có **{int(public_test['english_markdown'].isin(train_en).sum())}** document tiếng Anh trùng hoàn toàn.
- Phần lớn tài liệu cần được chia thành nhiều chunk theo cấu trúc pháp lý; không nên cắt cứng giữa điều/khoản.
- Bước tiếp theo hợp lý là xây **document-level split → legal segmenter → bilingual aligner → tokenizer analysis** cho từng pretrained model.
"""))


### Những điều cần nhớ

- Train có **854** document song ngữ; public test có **50** document.
- Độ dài tiếng Anh median là **29,166 ký tự**; document lớn nhất có **596,720 ký tự**.
- Có **34 lượt document** thuộc các nhóm cặp song ngữ trùng hoàn toàn. Nên deduplicate trước khi chia train/dev.
- Lĩnh vực lớn nhất trong train là **`Vi_pham_hanh_chinh`**. Khi tạo dev split nên stratify theo `category`.
- Train và public test có **0** document tiếng Anh trùng hoàn toàn.
- Phần lớn tài liệu cần được chia thành nhiều chunk theo cấu trúc pháp lý; không nên cắt cứng giữa điều/khoản.
- Bước tiếp theo hợp lý là xây **document-level split → legal segmenter → bilingual aligner → tokenizer analysis** cho từng pretrained model.
